[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/templates/18_embedding.ipynb)

# 🟢 Easy: Embedding Layer

*Core Ops & Layers*
Implement an **embedding table** as an `nnx.Module`.

Map integer token indices to dense vectors: `(...) -> (..., embedding_dim)`.

### Rules
- Signature: `MyEmbedding(num_embeddings, embedding_dim, *, rngs)`
- Table stored as `self.table`, an `nnx.Param` of shape `(num_embeddings, embedding_dim)`
- Initialise with `jax.random.normal(...) * 0.02` (the GPT-2 convention)
- `__call__(indices)` accepts **any** shape of integer indices
- Also implement `attend(x)`: `(..., embedding_dim) -> (..., num_embeddings)`,
  the transpose projection used for weight tying

### Indexing vs one-hot
These compute the same thing:

```python
table[indices]                        # gather
jax.nn.one_hot(indices, V) @ table    # matmul
```

The gather is `O(1)` per token; the matmul is `O(V)` per token and materialises a
`(B, T, V)` intermediate — with `V = 50257` that is enormous. Always gather.

(The one-hot form is not useless, though: on TPU it can be faster for small
vocabularies, and it is how you'd explain the *gradient*.)

### Why the gradient is sparse
`d(loss)/d(table)` is nonzero only at the rows you actually looked up, and
repeated indices **accumulate**. JAX handles this correctly through
`.at[].add()` semantics under the hood — but it produces a *dense* gradient array
with mostly zeros, which is why large-vocabulary models want sparse optimizer
support.

### Weight tying
`attend` exists because most language models share one matrix between the input
embedding and the output projection. It saves `V x d` parameters (about 40M for
GPT-2) and generally improves perplexity.

In [ ]:
# Colab setup (no-op when running locally).
# jax-judge is not published on PyPI, so the judge is installed from the
# repo itself. Regenerate with JAXCODE_REPO=you/YourFork to point this at
# your own fork:  JAXCODE_REPO=you/JAXCode make notebooks
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q flax optax')
    get_ipython().run_line_magic(
        'pip', 'install -q git+https://github.com/YOUR-GITHUB-USERNAME/JAXCode.git')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

import jax
import jax.numpy as jnp
from flax import nnx


class MyEmbedding(nnx.Module):
    """Integer indices -> dense vectors."""

    def __init__(self, num_embeddings: int, embedding_dim: int, *, rngs: nnx.Rngs):
        pass  # Replace this

    def __call__(self, indices):
        """(...) integer indices -> (..., embedding_dim)"""
        pass  # Replace this

    def attend(self, x):
        """(..., embedding_dim) -> (..., num_embeddings). Transpose projection."""
        pass  # Replace this

In [ ]:
# 🔍 Scratch cell — poke at your implementation
import jax.numpy as jnp
from flax import nnx

emb = MyEmbedding(100, 8, rngs=nnx.Rngs(params=0))

print("table:", emb.table.shape)
print("scalar id  ->", emb(jnp.array(5)).shape)
print("(3,) indices   ->", emb(jnp.array([1, 2, 3])).shape)
print("(2,4) indices  ->", emb(jnp.zeros((2, 4), dtype=jnp.int32)).shape)
print("attend     ->", emb.attend(jnp.ones((2, 4, 8))).shape)

In [ ]:
# ✅ SUBMIT — run this cell to check your solution
from jax_judge import check, hint, solution, status

check("embedding")

# hint("embedding")      # stuck? nudge without the answer
# solution("embedding")  # spoiler: the reference implementation
# status()               # your dashboard across all problems